# NoxTrader: LSTM-Based Stock Return Momentum Prediction for Quantitative Trading

Authors: Hsiang-Hui Liu, Han-Jay Shu, Wei-Ning Chiu
Published: 2023-10-01
ArXiv: [https://arxiv.org/abs/2310.00747](https://arxiv.org/abs/2310.00747)

## Strategy Description
We introduce NoxTrader, a system for portfolio construction and trading execution designed to achieve profitable outcomes in the stock market. The system uses historical price and volume data to generate features such as Return Momentum, Week Price Momentum, and Month Price Momentum. An LSTM model is employed to capture continuous price trends, with dynamic model updates to adapt to current market trends. The system aims to generate moderate to long-term profits through careful feature engineering and predictive scoring.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
RISK_FREE_RATE = 0.02
INITIAL_CAPITAL = 100000

# Hypothesis
"""
We hypothesize that by using LSTM to predict return momentum and incorporating week and month price momentum,
we can generate a profitable trading strategy that adapts to changing market conditions.
""

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

def download_data(universe, start_date, end_date):
    data = yf.download(universe, start=start_date, end=end_date, group_by='ticker')
    return data

def compute_features(data):
    features = pd.DataFrame()
    for ticker in data:
        df = data[ticker]
        df['Return Momentum'] = df['Close'].pct_change().rolling(window=5).mean()
        df['Week Price Momentum'] = df['Close'].pct_change(periods=5).rolling(window=4).mean()
        df['Month Price Momentum'] = df['Close'].pct_change(periods=21).rolling(window=4).mean()
        features[ticker] = df[['Return Momentum', 'Week Price Momentum', 'Month Price Momentum']]
    return features

# Download and compute features
data = download_data(UNIVERSE, START_DATE, END_DATE)
features = compute_features(data)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
from scipy.stats import zscore

def generate_signals(features):
    signals = features.apply(zscore).rank(axis=1).sum(axis=1).apply(lambda x: 1 if x > len(UNIVERSE)/2 else -1)
    return signals

def construct_portfolio(signals, data):
    weights = signals / signals.abs().sum()
    portfolio_returns = (data['Close'].pct_change() * weights).sum(axis=1)
    return portfolio_returns

# Generate signals and construct portfolio
signals = generate_signals(features)
portfolio_returns = construct_portfolio(signals, data)

## Phase 4 — Vectorized Backtest

In [ ]:
def backtest(portfolio_returns, initial_capital, risk_free_rate):
    portfolio_value = (portfolio_returns + 1).cumprod() * initial_capital
    excess_returns = portfolio_returns - risk_free_rate
    return portfolio_value, excess_returns

# Backtest
portfolio_value, excess_returns = backtest(portfolio_returns, INITIAL_CAPITAL, RISK_FREE_RATE)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gmean

def calculate_performance_metrics(portfolio_value, excess_returns):
    sharpe_ratio = np.mean(excess_returns) / np.std(excess_returns)
    sortino_ratio = np.mean(excess_returns) / np.std(excess_returns[excess_returns < 0])
    calmar_ratio = np.mean(portfolio_returns) / (portfolio_value.max() - portfolio_value.min())
    max_drawdown = (portfolio_value.cummax() - portfolio_value).max()
    
    print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
    print(f'Sortino Ratio: {sortino_ratio:.2f}')
    print(f'Calmar Ratio: {calmar_ratio:.2f}')
    print(f'Max Drawdown: {max_drawdown:.2f}')
    
    plt.figure(figsize=(10, 5))
    plt.plot(portfolio_value, label='Portfolio Value')
    plt.title('Equity Curve')
    plt.xlabel('Date')
    plt.ylabel('Portfolio Value')
    plt.legend()
    plt.show()

# Calculate performance metrics
calculate_performance_metrics(portfolio_value, excess_returns)

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_portfolio(data, signals):
    current_prices = data['Close'].iloc[-1]
    current_positions = signals * current_prices
    daily_pnl = (data['Close'].pct_change().iloc[-1] * current_positions).sum()
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(current_positions)

# Monitor portfolio
monitor_portfolio(data, signals)